### Leer todos los datos que son requeridos

In [0]:
%run "../../utilerias/funciones_comunes"

In [0]:
dbutils.widgets.text("param_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("param_file_date")


In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movies_df = spark.read.table("smartdata_proyect_silver.movies").filter(f"file_date = '{v_file_date}'")  

In [0]:
languages_df = spark.read.table("smartdata_proyect_silver.languages") 

In [0]:
movies_languages_df = spark.read.table("smartdata_proyect_silver.movies_languages").filter(f"file_date = '{v_file_date}'") 

In [0]:
genres_df = spark.read.table("smartdata_proyect_silver.genres") 

In [0]:
movies_genres_df = spark.read.table("smartdata_proyect_silver.movies_genres").filter(f"file_date = '{v_file_date}'")  

#### Join "languages" y "movies_languages"

In [0]:
# Join entre los dataframe languages_df y movies_languages_df
movies_languages_inner_df = languages_df.join(movies_languages_df, languages_df.language_id == movies_languages_df.language_id, 'inner')\
    .select(languages_df.language_name, movies_languages_df.movie_id) 

In [0]:
# Join entre los dataframes genres_df y movies_genres_df
movies_genres_inner_df = genres_df.join(movies_genres_df, genres_df.genre_id == movies_genres_df.genre_id, 'inner')\
    .select(genres_df.genre_name, movies_genres_df.movie_id)

#### Join "movies_df", "movies_languages_inner_df" y "movies_genres_inner_df"

In [0]:
movies_languages_genres_df = movies_df.join(movies_languages_inner_df, movies_df.movie_id == movies_languages_inner_df.movie_id, 'inner')\
    .join(movies_genres_inner_df, movies_df.movie_id == movies_genres_inner_df.movie_id, 'inner')\
    .select(movies_df.title, movies_df.duration_time, movies_df.release_date, movies_df.vote_average,movies_languages_inner_df.language_name, movies_genres_inner_df.genre_name)

#### Creando una nueva columna llamada "created_date"

In [0]:
movies_languages_genres_final_df = movies_languages_genres_df.withColumn('created_date', lit(v_file_date))

#### Ordenar por la columna "release_date" de manera desendente

In [0]:
delete_records_by_filter("movie_gold", "results_movie_genre_languages","created_date", v_file_date)

In [0]:
result_order_by_df = movies_languages_genres_final_df.orderBy(movies_languages_genres_final_df.release_date.desc())

#### Escribir datos en el Datalake en formato "Parquet"

In [0]:
result_order_by_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("smartdata_proyect_golden.results_movie_genre_languages")

In [0]:
%sql
select count(1) , created_date from smartdata_proyect_golden.results_movie_genre_languages group by created_date;

In [0]:
%sql
select * from smartdata_proyect_golden.results_movie_genre_languages;